# Notebook 03 — RAG Pipeline sobre Corpus Etiquetado

**Curso:** Minería de Textos | **Proyecto 3** | CUC

---

## IMPORTANTE: Este notebook se ejecuta DESPUÉS del Notebook 02

El RAG se construye sobre el corpus ya etiquetado por el clasificador fine-tuneado.
Cada chunk hereda la emoción asignada por el modelo, lo que permite filtrado preciso.

```
Notebook 02 → Fine-Tuning → etiquetar_corpus_con_modelo() → corpus_con_emociones.pkl
                                        ↓
Notebook 03 → Chunking + Embeddings + FAISS → RAG listo
```

In [1]:
!pip install -q pymongo[srv] sentence-transformers faiss-cpu transformers python-dotenv numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.append('../app')
sys.path.append('../src')

import numpy as np
import json
from pathlib import Path

from app.config import  TOP_K, FLAN_T5_MODEL, SYSTEM_PROMPT
from src.mongo_utils import mongo_utils
from src.finetuning_utils import finetuning_utils
from src.rag_utils import rag_utils
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print('Librerias cargadas.')

C:\Users\pmari\OneDrive\Para Revisar\Documentos\Pablo\Cuc\2026\Mineria de Textos\chatbot-musical-inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerias cargadas.


## 1. Cargar corpus etiquetado por el modelo fine-tuneado

In [3]:
db_conexion = mongo_utils()
finetuning = finetuning_utils()

if db_conexion.verificar_conexion():
   canciones_raw=db_conexion.cargar_canciones()
   canciones_etiquetadas = finetuning.etiquetar_corpus_con_modelo(canciones_raw)
   print(f'Canciones con emocion: {len(canciones_etiquetadas)}')
   # Verificar que tienen el campo emocion
   ejemplo = canciones_etiquetadas[0]
   print(f'Ejemplo: {ejemplo.get("titulo")} — Emocion: {ejemplo.get("emocion")} ({ejemplo.get("emocion_score",0):.0%})')
else:
   canciones_raw=[]
   print(f'Canciones no cargadas: {len(canciones_raw)}')



2026-04-20 00:13:48 | INFO     | mongo_utils | Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical
2026-04-20 00:13:50 | INFO     | mongo_utils | Conexión verificada correctamente.
2026-04-20 00:13:50 | INFO     | mongo_utils | Cargando canciones | limite=None | solo_con_letra=True
2026-04-20 00:13:52 | INFO     | mongo_utils | Canciones cargadas: 6940
2026-04-20 00:13:52 | INFO     | finetuning_utils | Cargando corpus etiquetado desde caché...
2026-04-20 00:13:52 | INFO     | finetuning_utils | 6940 canciones cargadas con emoción.
Canciones con emocion: 6940
Ejemplo: ​thank u, next — Emocion: amor (97%)


## 2. Comparación de estrategias de chunking

In [4]:
rag = rag_utils()
chunks_parrafos  = []
chunks_completos = []
for c in canciones_etiquetadas:
    chunks_parrafos.extend(rag.chunking_por_parrafos(c))
    chunks_completos.extend(rag.chunking_cancion_completa(c))

print('COMPARACION DE ESTRATEGIAS DE CHUNKING')
print('=' * 60)
for nombre, lista in [('Por parrafos/estrofas', chunks_parrafos),
                       ('Cancion completa',      chunks_completos)]:
    tamanos = [len(c['texto']) for c in lista]
    print(f'\n{nombre}:')
    print(f'  Total chunks:    {len(lista)}')
    print(f'  Promedio chars:  {np.mean(tamanos):.0f}')
    print(f'  Min / Max:       {min(tamanos)} / {max(tamanos)}')
    print(f'  Ejemplo texto:   "{lista[0]["texto"][:80]}..."')
    print(f'  Emocion:         {lista[0]["emocion"]} ({lista[0]["emocion_score"]:.0%})')
    print(f'  Metadatos:       {lista[0]["titulo"]} - {lista[0]["artista"]} ({lista[0]["genero"]})')

COMPARACION DE ESTRATEGIAS DE CHUNKING

Por parrafos/estrofas:
  Total chunks:    6940
  Promedio chars:  2102
  Min / Max:       92 / 29720
  Ejemplo texto:   "thought i'd end up with sean but he wasn't a match wrote some songs about ricky ..."
  Emocion:         amor (97%)
  Metadatos:       ​thank u, next - Ariana Grande (pop)

Cancion completa:
  Total chunks:    6940
  Promedio chars:  1600
  Min / Max:       92 / 2000
  Ejemplo texto:   "thought i'd end up with sean but he wasn't a match wrote some songs about ricky ..."
  Emocion:         amor (97%)
  Metadatos:       ​thank u, next - Ariana Grande (pop)


## 3. Construir indice FAISS sobre corpus etiquetado

In [5]:
indice, chunks_activos = rag.inicializar(canciones_etiquetadas)
print(f'\nChunks indexados: {len(chunks_activos)}')
print(f'Ejemplo de chunk con emocion:')
ej = chunks_activos[0]
print(f'  Titulo:  {ej["titulo"]}')
print(f'  Emocion: {ej["emocion"]} ({ej["emocion_score"]:.0%})')
print(f'  Texto:   {ej["texto"][:100]}...')

2026-04-20 00:14:03 | INFO     | rag_utils | Construyendo RAG desde corpus etiquetado...
2026-04-20 00:14:03 | INFO     | rag_utils | Construyendo chunks para 6940 canciones...
2026-04-20 00:14:03 | INFO     | rag_utils | Chunks generados: 6940
2026-04-20 00:14:03 | INFO     | rag_utils | Generando embeddings para 6940 chunks...
2026-04-20 00:14:03 | INFO     | rag_utils | Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4134.94it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-20 00:14:10 | INFO     | rag_utils | Modelo cargado. Dimensión: 384


C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\src\rag_utils.py:165: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self._modelo_emb.get_sentence_embedding_dimension()
Batches: 100%|██████████| 109/109 [00:48<00:00,  2.24it/s]

2026-04-20 00:14:59 | INFO     | rag_utils | Embeddings guardados: (6940, 384)
2026-04-20 00:14:59 | INFO     | rag_utils | Construyendo índice FAISS...


2026-04-20 00:14:59 | INFO     | rag_utils | Índice guardado: 6940 vectores
2026-04-20 00:14:59 | INFO     | rag_utils | RAG listo: 6940 chunks indexados.

Chunks indexados: 6940
Ejemplo de chunk con emocion:
  Titulo:  ​thank u, next
  Emocion: amor (97%)
  Texto:   thought i'd end up with sean but he wasn't a match wrote some songs about ricky now i listen and lau...


## 4. Pruebas de busqueda semantica con filtro por emocion

In [6]:
pruebas = [
    ('canciones sobre tristeza y desamor', 'tristeza'),
    ('musica alegre para una fiesta',      'alegria'),
    ('letras de amor romantico',           'amor'),
    ('canciones de rabia y traicion',      'rabia'),
]

for pregunta, emocion_filtro in pruebas:
    print(f'\nPregunta: {pregunta}')
    print(f'Filtro emocion: {emocion_filtro}')
    print('-' * 50)
    # Con filtro de emocion del clasificador fine-tuneado
    resultados = rag.buscar(pregunta, top_k=3, filtro_emocion=emocion_filtro)
    if not resultados:
        print('  Sin resultados con filtro, buscando sin filtro...')
        resultados = rag.buscar(pregunta, top_k=3)
    for i, r in enumerate(resultados):
        print(f'  [{i+1}] Score: {r["score"]:.4f} | Emocion: {r["emocion"]:10s} | {r["titulo"]} - {r["artista"]}')
        print(f'       {r["texto"][:80]}...')


Pregunta: canciones sobre tristeza y desamor
Filtro emocion: tristeza
--------------------------------------------------
  [1] Score: 0.4088 | Emocion: tristeza   | They ask you how you are, and you just have to say you’re fine - Katy Perry
       you know sometimes you can be blinded by your extreme emotions i definitely was ...
  [2] Score: 0.3491 | Emocion: tristeza   | They ask you how you are, and you just have to say you’re fine - Katy Perry
       "You know, sometimes you can be blinded by your extreme emotions. I definitely w...
  [3] Score: 0.3259 | Emocion: tristeza   | let’s pretend we’re numb - XXXTentacion
       As a fair warning to everyone listening to this song I advise you to not hide yo...

Pregunta: musica alegre para una fiesta
Filtro emocion: alegria
--------------------------------------------------
  [1] Score: 0.2114 | Emocion: alegria    | Mi Gente (Beyoncé Remix) - J Balvin
       ¡Wuh! Si el ritmo te lleva a mover la cabeza Ya empezamos cómo es Mi música no

## 5. Generador Flan-T5 local

In [7]:
device = 0 if torch.cuda.is_available() else -1
dispositivo = "cuda" if device == 0 else "cpu"
print(f'Dispositivo: {dispositivo.upper()}')

print(f'Cargando {FLAN_T5_MODEL}...')
flan_tokenizer = AutoTokenizer.from_pretrained(FLAN_T5_MODEL)
flan_model     = AutoModelForSeq2SeqLM.from_pretrained(FLAN_T5_MODEL).to(dispositivo)

def generador(prompt, max_new_tokens=300):
    inputs = flan_tokenizer(
        prompt[0] if isinstance(prompt, list) else prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(dispositivo)
    outputs = flan_model.generate(**inputs, max_new_tokens=max_new_tokens)
    texto = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": texto}]

print('Flan-T5 listo.')

Dispositivo: CUDA
Cargando google/flan-t5-base...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1661.10it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Flan-T5 listo.


## 6. Comparacion CON vs SIN RAG

In [8]:

def rag_completo_nb(pregunta, top_k=3, filtro_emocion=None):
    chunks = rag.buscar(pregunta, top_k=top_k, filtro_emocion=filtro_emocion)
    if not chunks:
        chunks = rag.buscar(pregunta, top_k=top_k)
    ctx = '\n\n'.join(
        f'[{c["titulo"]} - {c["artista"]} | {c["genero"]} | {c["anio"]} | Emocion: {c["emocion"]}]\n{c["texto"]}'
        for c in chunks
    )
    prompt = f'{SYSTEM_PROMPT}\n\nFragmentos:\n{ctx}\n\nPregunta: {pregunta}\nRespuesta:'
    respuesta = generador(prompt)[0]['generated_text'].strip()
    return respuesta, chunks

def sin_rag(pregunta):
    prompt = f'Eres experto en musica. Pregunta: {pregunta}\nRespuesta:'
    return generador(prompt)[0]['generated_text'].strip()

preguntas_comp = [
    ('Que canciones hablan de desamor?',           'tristeza'),
    ('Que cancion recomiendas para alguien triste?', 'tristeza'),
    ('Dame una cancion de pop alegre.',              'alegria'),
]

resultados_comp = []
for pregunta, emocion in preguntas_comp:
    print(f'\n{"="*60}')
    print(f'PREGUNTA: {pregunta} | Emocion filtro: {emocion}')
    resp_sin = sin_rag(pregunta)
    resp_con, chunks = rag_completo_nb(pregunta, top_k=3, filtro_emocion=emocion)
    print(f'SIN RAG: {resp_sin}')
    print(f'CON RAG: {resp_con}')
    print(f'Fuentes: {[c["titulo"] + " (" + c["emocion"] + ")" for c in chunks]}')
    resultados_comp.append({'pregunta': pregunta, 'sin_rag': resp_sin, 'con_rag': resp_con,
        'fuentes': [{'titulo': c['titulo'], 'artista': c['artista'], 'emocion': c['emocion'],
                     'score': c['score']} for c in chunks]})

Path('../resultados').mkdir(exist_ok=True)
with open('../resultados/comparacion_rag.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_comp, f, ensure_ascii=False, indent=2)
print('\nResultados guardados en resultados/comparacion_rag.json')


PREGUNTA: Que canciones hablan de desamor? | Emocion filtro: tristeza
SIN RAG: Te es experto en msica?
CON RAG: ok
Fuentes: ['They ask you how you are, and you just have to say you’re fine (tristeza)', 'let’s pretend we’re numb (tristeza)', 'They ask you how you are, and you just have to say you’re fine (tristeza)']

PREGUNTA: Que cancion recomiendas para alguien triste? | Emocion filtro: tristeza
SIN RAG: Te es tan sabio?
CON RAG: that song and yeah i was depressed and sad and there were thoughts but there were never actions thankfully but i wanted to share that side of my story because i know there are so many other people out there that have gone through things like that and you always feel like you’re the only one going through that you walk out the door and you see someone you know and they ask you how you are and you just have to say youre fine when you’re not really fine but you just can’t get into it because they would never understand well then comes along a song that speaks 